# 04 · A Codebase-Explainer Agent — with a Gradio UI

**Where we are in the stack:** still the **Agent = control plane** (Layer 4),
exactly like notebook 2 - but now the tools touch a *real* resource: the
**filesystem**. The agent answers questions about a project by *exploring its
source*, the way you would in a terminal.

Same loop you already know:

```
prompt the model  ->  did it ask for a tool?
      ^                       |  yes                 |  no
      |                  call the tool          FINAL ANSWER -> stop
      |                       |
      +----- feed result back +
```

Input: a **source path** + a **question**.
Output: an explanation the model assembled by running `ls`, `cat`, `grep`, `find`
on the code itself - it is not allowed to guess from memory.

> Tools are **read-only**. The agent can look, not touch. This is the safe
> default from notebook 2, enforced here for filesystem access.

## 0. Provider config (identical to the other notebooks)

Same env-var block: `OPENAI_BASE_URL`, `OPENAI_API_KEY`, `MODEL`.
Needs a **tool-capable** model (`gpt-4o-mini`, local `qwen2.5` / `llama3.1`).

In [ ]:
# --- Provider config: works with OpenAI, OpenRouter, or a local OpenAI-compatible server ---
import os
from openai import OpenAI

# Load settings from a .env file if present (falls back to existing env vars).
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except Exception:
    import os
    if os.path.exists(".env"):
        for _line in open(".env"):
            _line = _line.strip()
            if _line and not _line.startswith("#") and "=" in _line:
                _k, _v = _line.split("=", 1)
                os.environ.setdefault(_k.strip(), _v.strip())


BASE_URL = os.environ.get("OPENAI_BASE_URL", "https://api.openai.com/v1")
API_KEY  = os.environ.get("OPENAI_API_KEY", "set-me")   # any non-empty string for local servers
MODEL    = os.environ.get("MODEL", "gpt-4o-mini")

client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
print("endpoint:", BASE_URL, "| model:", MODEL)

## 1. The tools = a tiny, read-only terminal

Five plain Python functions, each one a familiar shell command. Two rules keep
them safe and demo-friendly:

- **Sandboxed**: every path is resolved and must stay inside the project root
  (`ROOT`). No escaping with `..` or absolute paths.
- **Read-only**: list, read, search. Nothing writes, moves, or deletes.

`ROOT` is the one piece of state the agent operates over - set it to the source
path you want explained.

In [ ]:
import os, subprocess

# The project the agent is allowed to explore. Point this at any source tree.
ROOT = os.path.abspath(".")

def _safe(path):
    """Resolve `path` under ROOT and refuse anything that escapes the sandbox."""
    full = os.path.abspath(os.path.join(ROOT, path))
    if full != ROOT and not full.startswith(ROOT + os.sep):
        raise ValueError(f"path '{path}' escapes the project root")
    return full

def list_dir(path="."):
    """ls: list files/dirs at `path` (relative to the project root)."""
    full = _safe(path)
    if not os.path.isdir(full):
        return {"error": f"not a directory: {path}"}
    entries = []
    for name in sorted(os.listdir(full)):
        p = os.path.join(full, name)
        entries.append(name + ("/" if os.path.isdir(p) else ""))
    return {"path": path, "entries": entries}

def read_file(path, max_bytes=4000):
    """cat: read a text file (truncated to max_bytes so we don't blow the context)."""
    full = _safe(path)
    if not os.path.isfile(full):
        return {"error": f"not a file: {path}"}
    with open(full, "r", errors="replace") as f:
        data = f.read(max_bytes + 1)
    truncated = len(data) > max_bytes
    return {"path": path, "content": data[:max_bytes], "truncated": truncated}

def grep(pattern, path=".", max_matches=40):
    """grep -rn: search file contents for `pattern`, return file:line:text matches."""
    full = _safe(path)
    out = subprocess.run(
        ["grep", "-rIn", "--", pattern, full],
        capture_output=True, text=True,
    ).stdout
    matches = []
    for line in out.splitlines()[:max_matches]:
        matches.append(line.replace(ROOT + os.sep, "").replace(ROOT, "."))
    return {"pattern": pattern, "matches": matches, "count": len(matches)}

def find_files(name_glob="*", path="."):
    """find -name: list files whose name matches a glob (e.g. '*.py')."""
    full = _safe(path)
    out = subprocess.run(
        ["find", full, "-type", "f", "-name", name_glob],
        capture_output=True, text=True,
    ).stdout
    files = [l.replace(ROOT + os.sep, "").replace(ROOT, ".")
             for l in out.splitlines()]
    return {"name_glob": name_glob, "files": sorted(files)[:80]}

# Sanity-check the tools WITHOUT the model (prove they work first)
print(list_dir("."))
print(find_files("*.ipynb"))

## 2. Describe the tools to the model (JSON schema)

The model never runs these. It only *requests* a call by name + JSON args;
**you** execute it and feed the result back. This is the same capability-table
pattern from notebook 2.

In [ ]:
TOOLS = [
    {"type": "function", "function": {
        "name": "list_dir",
        "description": "List files and subdirectories at a path (relative to project root). Like `ls`.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string", "description": "dir path, default '.'"}},
            "required": []}}},
    {"type": "function", "function": {
        "name": "read_file",
        "description": "Read the contents of a text file (truncated). Like `cat`.",
        "parameters": {"type": "object",
            "properties": {"path": {"type": "string", "description": "file path relative to root"}},
            "required": ["path"]}}},
    {"type": "function", "function": {
        "name": "grep",
        "description": "Search file contents recursively for a pattern. Returns file:line:text. Like `grep -rn`.",
        "parameters": {"type": "object",
            "properties": {
                "pattern": {"type": "string", "description": "text/regex to search for"},
                "path":    {"type": "string", "description": "dir to search, default '.'"}},
            "required": ["pattern"]}}},
    {"type": "function", "function": {
        "name": "find_files",
        "description": "List files whose name matches a glob, e.g. '*.py'. Like `find -name`.",
        "parameters": {"type": "object",
            "properties": {
                "name_glob": {"type": "string", "description": "glob like '*.py'"},
                "path":      {"type": "string", "description": "dir to search, default '.'"}},
            "required": ["name_glob"]}}},
]

# name -> callable. The agent's capability table.
TOOL_REGISTRY = {
    "list_dir": list_dir,
    "read_file": read_file,
    "grep": grep,
    "find_files": find_files,
}

## 3. The agent loop (the same FSM as notebook 2)

Nothing here is new - prompt, check for a tool request, run it, feed the result
back, repeat until the model gives a final answer or we hit the TTL. The only
difference from notebook 2 is the *system prompt*, which tells the model how to
explore code well.

In [ ]:
import json

SYSTEM = """You are a codebase-explainer agent.
You answer questions about a project ONLY by inspecting its source with your tools.
Strategy:
  1. Start with list_dir('.') and read any README / entry points.
  2. Use find_files and grep to locate the relevant code before reading it.
  3. Read the specific files that matter; do not guess.
Ground every claim in something you actually saw. Cite file paths.
Be concise and structured. If unsure, look - do not speculate."""

def explain(question, max_iterations=12, temperature=0):
    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": f"Project root: {ROOT}\n\nQuestion: {question}"},
    ]
    print("ROOT:", ROOT)
    print("Q:", question)
    print("=" * 72)

    for step in range(1, max_iterations + 1):
        resp = client.chat.completions.create(
            model=MODEL, messages=messages, tools=TOOLS, temperature=temperature,
        )
        msg = resp.choices[0].message

        # TERMINATION: no tool requested -> final answer.
        if not msg.tool_calls:
            print(f"\n[step {step}] FINAL ANSWER\n{'-'*72}\n{msg.content}")
            return msg.content

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {"id": tc.id, "type": "function",
                 "function": {"name": tc.function.name, "arguments": tc.function.arguments}}
                for tc in msg.tool_calls
            ],
        })

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"[step {step}] TOOL  -> {name}({args})")
            try:
                result = TOOL_REGISTRY[name](**args)         # <-- WE run it
            except Exception as e:
                result = {"error": str(e)}
            preview = str(result)
            print(f"[step {step}] RESULT <- {preview[:200]}{'...' if len(preview) > 200 else ''}")
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result)[:4000],
            })

    return "Stopped: hit max_iterations (TTL expired)."

## 4. Run it - explain *this* project

Point `ROOT` at any source tree. Here we aim it at the current folder so the
agent explains these very notebooks.

In [ ]:
ROOT = os.path.abspath(".")   # explain this repo; change to any project path
_ = explain("What is this project and what does each file do? Give a short tour.")

## 5. Run it - a targeted question

Watch the agent `grep`/`find` its way to the answer instead of dumping whole
files.

In [ ]:
_ = explain("How is the agent loop implemented, and where does it decide to stop?")

## 6. Point it at a different codebase

The agent is generic - give it any path and any question.

```python
ROOT = os.path.abspath("/path/to/some/project")
_ = explain("What are the main components and how do they fit together?")
```

## Recap

You built a second real agent with **the identical loop** from notebook 2 - only
the tools changed. That is the point: an agent is *a loop + a capability table*.
Swap network tools for filesystem tools and the same control plane now explains
code.

Production-grade versions of this (Cursor, Copilot, Claude Code) add what
notebook 2 already listed as missing: retries, parallel tool calls, streaming,
persistent memory/checkpointing, observability, and confirmation gates before
any *write* tool. Here we stayed **read-only and sandboxed** on purpose.

**The whole arc:** 01 completion -> 02 hand-rolled agent -> 03 framework -> 04
the same agent pointed at a new resource. Tools are how the model stops *talking*
and starts *doing*.

## 7. Interactive UI (Gradio)

Point the agent at any project path and ask a question. The **Agent trace** box
shows it `ls`/`grep`/`cat` its way to the answer.

In [ ]:
# Install Gradio for the interactive UI (run once).
%pip install -q gradio

In [ ]:
import io, contextlib

def _run_capture(fn, *args, **kwargs):
    """Run an agent function, capturing its printed step-by-step trace."""
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        try:
            result = fn(*args, **kwargs)
        except Exception as e:
            result = f"Error: {e}"
    return buf.getvalue(), (result or "")

In [ ]:
import gradio as gr

def explain_ui(project_path, question):
    global ROOT
    ROOT = os.path.abspath(project_path or ".")
    trace, answer = _run_capture(explain, question)
    return trace, answer

with gr.Blocks(title="04 · Codebase Explainer") as demo:
    gr.Markdown("# 📂 Codebase-Explainer Agent\n"
                "Read-only, sandboxed filesystem tools (ls / cat / grep / find). "
                "Give it a path and a question; it explores the source to answer.")
    project_path = gr.Textbox(label="Project path", value=".")
    question = gr.Textbox(
        label="Question",
        value="What is this project and what does each file do? Give a short tour.",
    )
    run = gr.Button("Explain", variant="primary")
    gr.Examples(
        ["What is this project and what does each file do? Give a short tour.",
         "How is the agent loop implemented, and where does it decide to stop?",
         "What are the main components and how do they fit together?"],
        inputs=question,
    )
    answer = gr.Markdown(label="Explanation")
    trace = gr.Textbox(label="Agent trace (tool calls)", lines=16)
    run.click(explain_ui, inputs=[project_path, question], outputs=[trace, answer])

demo.launch()